In [104]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline,make_pipeline
import scipy.stats as stats
from sklearn.compose import ColumnTransformer
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

In [111]:
df = pd.read_csv('datasets/laptops_cleaned.csv')

In [112]:
df.head()

,Brand,processor_brand,processor_series,processor_tier,ram,storage,screen_size,display_type,gpu_brand,gpu_model,vram,prices
0,Lenovo,Intel,Core Ultra,7,32,1TB,16.0,WQXGA,Integrated,Integrated,0,214999
1,Dell,Intel,Core,7,16,1TB,16.0,WQXGA,Integrated,Integrated,0,159990
2,Acer,AMD,Ryzen,3,8,512GB,15.6,FHD,Integrated,Integrated,0,52599
3,Samsung,Intel,Core Ultra,7,16,512GB,16.0,WUXGA,Integrated,Integrated,0,131990
4,Samsung,Intel,Core Ultra,5,16,512GB,16.0,WUXGA,Integrated,Integrated,0,123990


In [ ]:
def apply_transform(transform):
    X_temp = df
 
    
    trf = ColumnTransformer([('log',FunctionTransformer(transform),['prices'])],remainder='passthrough')
    
    X_trans = trf.fit_transform(X_temp)
    
    
    plt.figure(figsize=(14,4))

    plt.subplot(121)
    sns.histplot(X_temp['prices'], kde=True)
    plt.title('Fare Before Transform')

    plt.subplot(122)
    sns.histplot(X_trans[:,0], kde=True)
    plt.title('Fare After Transform')

    plt.show()
    print(skew(pd.to_numeric(X_temp['prices'])))
    print(skew(pd.to_numeric(X_trans[:,0])))
    
apply_transform(np.log)

In [107]:
X = df.drop(columns=['prices'])
y = np.log1p(df['prices'])

In [108]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [46]:
X_train

,Brand,processor_brand,processor_series,processor_tier,ram,storage,screen_size,display_type,gpu_brand,gpu_model,vram
113,Acer,Intel,Core i,7,16,1TB,16.0,WQXGA,Integrated,Integrated,0
101,Dell,AMD,Ryzen AI,5,16,512GB,16.0,2K,Integrated,Integrated,0
137,ASUS,Intel,Core Ultra,9,32,2TB,16.0,2.5K,Integrated,Integrated,0
73,Dell,Intel,Core i,5,16,1TB,15.6,FHD,Integrated,Integrated,0
93,Lenovo,Intel,Core i,5,16,512GB,15.6,FHD,Integrated,Integrated,0
...,...,...,...,...,...,...,...,...,...,...,...
188,ASUS,AMD,Ryzen,3,8,512GB,14.0,FHD,Integrated,Integrated,0
71,Lenovo,AMD,Ryzen,5,16,512GB,15.6,FHD,Integrated,Integrated,0
106,HP,Intel,Core i,7,24,1TB,15.6,FHD,Integrated,Integrated,0
270,Lenovo,Intel,Core i,5,12,512GB,15.6,FHD,NVIDIA,RTX2050,4


In [47]:
y_train

113    12.111713
101    11.197543
137    12.793834
73     11.225243
93     11.270854
         ...    
188    10.691740
71     10.896739
106    11.695172
270    11.082004
102    11.277089
Name: prices, Length: 251, dtype: float64

In [48]:
X_train.shape, y_train.shape

((251, 11), (251,))

## Basic structure without pipeline

In [49]:
scaler = StandardScaler()

X_train[['screen_size']] = scaler.fit_transform(
    X_train[['screen_size']]
)

X_test[['screen_size']] = scaler.transform(
    X_test[['screen_size']]
)

In [51]:
df['ram'].value_counts()

ram
16    215
8      39
24     28
32     27
12      2
48      2
36      1
Name: count, dtype: int64

In [52]:
oe = OrdinalEncoder(categories=[['0', '3', '5', '7', '9'],['8', '12', '16', '24', '32', '36', '48'], ['256GB', '512GB', '1TB', '2TB'], ['0', '4', '6', '8']])

In [53]:
X_train[['processor_tier', 'ram', 'storage', 'vram']] = oe.fit_transform(
    X_train[['processor_tier', 'ram', 'storage', 'vram']]
)

X_test[['processor_tier', 'ram', 'storage', 'vram']] = oe.transform(
    X_test[['processor_tier', 'ram', 'storage', 'vram']]
)

In [54]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 251 entries, 113 to 102
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Brand             251 non-null    object 
 1   processor_brand   251 non-null    object 
 2   processor_series  251 non-null    object 
 3   processor_tier    251 non-null    float64
 4   ram               251 non-null    float64
 5   storage           251 non-null    float64
 6   screen_size       251 non-null    float64
 7   display_type      251 non-null    object 
 8   gpu_brand         251 non-null    object 
 9   gpu_model         251 non-null    object 
 10  vram              251 non-null    float64
dtypes: float64(5), object(6)
memory usage: 23.5+ KB


In [56]:
from sklearn.preprocessing import OneHotEncoder


ohe_cols = ['Brand', 'processor_brand', 'processor_series', 'display_type', 'gpu_brand', 'gpu_model']

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first'), ohe_cols)
], remainder='passthrough')

X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

C:\Users\Himanshu\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [61]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

lr = LinearRegression()
scores = cross_val_score(lr, X_train, y_train, cv=5, scoring='r2')
print(scores.mean())

0.8771858431271777


In [62]:
lr.fit(X_train, y_train)
print(lr.score(X_test, y_test))

0.8451392390357014


## Creating pipeline

In [95]:
ohe_cols = ['Brand', 'processor_brand', 'processor_series', 'display_type', 'gpu_brand', 'gpu_model']
oe_cols = ['processor_tier', 'ram', 'storage', 'vram']
num_cols = ['screen_size']

preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), num_cols),
    ('oe', OrdinalEncoder(categories=[
        ['0', '3', '5', '7', '9'],
        ['8', '12', '16', '24', '32', '36', '48'],
        ['256GB', '512GB', '1TB', '2TB'],
        ['0', '4', '6', '8', '12']
    ]), oe_cols),
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first'), ohe_cols)
], remainder='passthrough')

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

In [96]:
scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')
print(scores.mean())

0.8771858431271777


C:\Users\Himanshu\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Himanshu\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## Try

In [101]:

input_data = pd.DataFrame([{
    'Brand': 'Lenovo',
    'processor_brand': 'Intel',
    'processor_series': 'Core Ultra',
    'processor_tier': '7',
    'ram': '32',
    'storage': '1TB',
    'screen_size': 16.0,
    'display_type': 'WQXGA',
    'gpu_brand': 'Integrated',
    'gpu_model': 'Integrated',
    'vram': '0'
}])

pipe.fit(X_train, y_train)
prediction = pipe.predict(input_data)

In [102]:
np.expm1(prediction)

array([220032.74678558])

In [109]:
import pickle

pickle.dump(pipe, open('model.pkl', 'wb'))